# Experimento SVM: Predicción Multiclase de Deserción Estudiantil (SPADIES - 3 Clases)
**Autor:** Arnold Santiago Torres | **Machine Learning II (2026-II)**

### Objetivos e Innovaciones Integradas (100% SVM - Norma SPADIES Colombia):
1. **Variable Objetivo Multiclase (SPADIES 3.0)**: Clasificación formal en 3 estados educativos: `Graduado`, `En curso` y `Desertor`, siguiendo el marco del Sistema de Prevención de la Deserción en las Instituciones de Educación Superior (SPADIES - MEN Colombia).
2. **Formulación Transparente del Índice de Riesgo ($R_i \in [0, 7.5]$)**: Índice multidimensional sintético con ponderación explícita de factores académicos, socioeconómicos e institucionales.
3. **Análisis Exploratorio de Datos (EDA) Ampliado**: Diagnóstico de *Outliers* mediante Rango Intercuartílico (IQR), asimetría (skewness) y matrices de correlación de Pearson y Spearman.
4. **Evaluación de Escaladores sobre Variables Sesgadas**: Visualización del impacto de `StandardScaler`, `MinMaxScaler`, `RobustScaler`, `QuantileTransformer` y `PowerTransformer`.
5. **Evaluación Multiclase de SVM sin Fuga de Datos (*Data Leakage*)**: 50 particiones aleatorias independientes (50 trials / 400 modelos entrenados) utilizando `ColumnTransformer` + `Pipeline` de Scikit-Learn.
6. **Inspección del Espacio Latente 2D y Matriz $3 \times 3$**: Visualización del hiperplano de separación vía PCA y auditoría por subgrupos socioeconómicos.

In [ ]:
import os
import sys
import warnings
from pathlib import Path

# Agregar raiz del proyecto al sys.path para importar src
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import StandardScaler

from src.data import (
    compute_outlier_diagnostics,
    create_spadies_target,
    get_train_test_split,
    load_data
)
from src.models import (
    compute_permutation_importance,
    fit_best_pipeline,
    get_normalization_scalers,
    run_multiclass_trials,
    save_model
)
from src.utils import (
    CLASSES,
    CLASS_MAP,
    FIGURES_DIR,
    METRICS_DIR,
    MODELS_DIR,
    NUM_COLS,
    compute_classification_metrics,
    compute_subgroup_recall,
    export_summary_to_csv
)
from src.visualization import (
    plot_confusion_matrices,
    plot_eda_distributions,
    plot_key_variables_distribution,
    plot_normalization_comparison,
    plot_pca_hyperplane,
    plot_permutation_importance,
    plot_scaler_impact
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['font.sans-serif'] = 'Arial'

print("✓ Paquetes modulares de 'src' importados y entorno configurado correctamente.")


--- 
## 1. Origen y Construcción de la Variable Objetivo Multiclase (SPADIES - 3 Clases)

Siguiendo la metodología del **Sistema de Prevención de la Deserción en Educación Superior (SPADIES 3.0)** de Colombia, definimos el estado estudiantil en 3 categorías mutuamente excluyentes ($y \in \{\text{'Desertor'}, \text{'En curso'}, \text{'Graduado'}\}$):

### 1. Índice de Riesgo Ponderado ($R_i \in [0, 7.5]$):
$$R_i = 2.5 \cdot \mathbb{I}(\text{promedio} < 3.0) + 2.5 \cdot \mathbb{I}(\text{reprobadas} \ge 2) + 1.0 \cdot \mathbb{I}(\text{icfes} < 40) + 1.0 \cdot \mathbb{I}(\text{trabaja} = 1) + 0.5 \cdot \mathbb{I}(\text{beca} = 0) + 0.5 \cdot \mathbb{I}(\text{estrato} \le 2)$$

### 2. Reglas de Clasificación SPADIES 3.0:
- **Desertor**: $R_i \ge 3.5$.
- **Graduado**: $R_i < 3.5$ y `semestre_cursado` $\ge$ Duración Nominal del Programa y `promedio_academico` $\ge 3.2$.
- **En curso**: Estudiante activo que no cumple condición de graduado ni de desertor.

In [ ]:
# Carga de datos y construccion de la variable objetivo SPADIES (3 clases)
raw_df = load_data()
df = create_spadies_target(raw_df)

print(f"Dimensiones del dataset: {df.shape[0]} estudiantes y {df.shape[1]} variables.")
print(f"Valores nulos totales en el dataset: {df.isnull().sum().sum()}")

print('\n=== DISTRIBUCION DE LA VARIABLE OBJETIVO (3 CLASES - SPADIES) ===')
dist = df['estado_final'].value_counts()
dist_pct = df['estado_final'].value_counts(normalize=True) * 100
dist_df = pd.DataFrame({'Estudiantes': dist, 'Porcentaje': dist_pct.round(2).astype(str) + '%'})
display(dist_df)

# Diagnostico estadistico de Outliers via IQR (modulo src.data)
print('\n=== DIAGNOSTICO DE OUTLIERS Y ASIMETRIA (VARIABLES NUMERICAS) ===')
diag_df = compute_outlier_diagnostics(df, NUM_COLS)
display(diag_df.style.format({
    'Media': '{:.2f}', 'Mediana': '{:.2f}', 'IQR': '{:.2f}',
    'Outliers %': '{:.2f}%', 'Skewness': '{:.2f}'
}))


## 2. Análisis Exploratorio de Datos (EDA) y Demostración de Escaladores

In [ ]:
# Graficos EDA: Distribucion de 3 Clases + Correlaciones Pearson vs Spearman (modulo src.visualization)
fig_eda, axes_eda = plot_eda_distributions(df)
plt.show()

# Distribucion de variables clave por estado final
fig_box, axes_box = plot_key_variables_distribution(df)
plt.show()


In [ ]:
# Demostracion del impacto de escaladores sobre variable con asimetria (modulo src.visualization)
fig_scalers, axes_scalers = plot_scaler_impact(df, feature='distancia_hogar_ies_km')
plt.show()


## 3. Experimento de Pipelines SVM Multiclase sin Fuga de Datos (50 Trials / 400 Modelos)

In [ ]:
# Preparacion de particiones sin fuga de datos y ejecucion de 50 trials independientes
X_train, X_test, y_train, y_test, X_raw, y = get_train_test_split(df, test_size=0.2, random_state=42)

N_TRIALS = 50
N_JOBS = -1

print(f'Iniciando {N_TRIALS} iteraciones paralelas de Pipelines Multiclase sin fuga de datos...')
results_weighted, results_macro, results_desertor, summary_df = run_multiclass_trials(
    X_raw=X_raw,
    y=y,
    n_trials=N_TRIALS,
    n_jobs=N_JOBS,
    test_size=0.2
)
print('✓ Iteraciones multiclase completadas exitosamente.')

# Guardar metricas en resultados/metricas/
export_summary_to_csv(summary_df)


In [ ]:
# Procesamiento e informe estadistico de los 50 trials multiclase
print("=== TABLA COMPARATIVA: PROMEDIO DE F1-SCORE MULTICLASE ===")
display(summary_df.style.format({
    'Average F1 (Weighted)': '{:.4f}',
    'Average F1 (Macro)': '{:.4f}',
    'Average F1 (Desertor)': '{:.4f}',
    'Std Dev': '{:.4f}',
    'F1 Delta vs Baseline': '{:+.4f}',
    'T-Statistic': '{:.3f}',
    'P-value vs Baseline': '{:.6f}'
}))

# Grafico comparativo de normalizaciones vs Baseline
fig_norm, ax_norm = plot_normalization_comparison(summary_df, n_trials=N_TRIALS)
plt.show()


## 4. Visualización del Espacio Latente 2D (PCA) y Matriz de Confusión Multiclase ($3 \times 3$)

In [ ]:
# Ajuste del pipeline optimo y serializacion de artefacto reproducible
best_pipeline = fit_best_pipeline(X_train, y_train, scaler=StandardScaler(), probability=False)
model_path = save_model(best_pipeline)
print(f"✓ Modelo serializado guardado en: {model_path}")

# Permutation Importance sobre el conjunto de prueba
print('\n=== TOP VARIABLES MAS INFLUYENTES (PERMUTATION IMPORTANCE) ===')
imp_df = compute_permutation_importance(best_pipeline, X_test, y_test, n_repeats=5, random_state=42)
display(imp_df.head(10))

fig_imp, ax_imp = plot_permutation_importance(imp_df, top_n=15)
plt.show()

# Visualizacion del Hiperplano SVM en Espacio Latente (PCA 2D)
fig_pca, ax_pca = plot_pca_hyperplane(best_pipeline, X_train, y_train, X_test, y_test)
plt.show()

# Reporte de Clasificacion Detallado y Matriz de Confusion
y_pred = best_pipeline.predict(X_test)
metrics = compute_classification_metrics(y_test, y_pred, target_names=CLASSES)

print('=== REPORTE DE CLASIFICACION MULTICLASE (3 CLASES SPADIES) ===')
print(metrics['classification_report'])
print(f"Accuracy Total:      {metrics['accuracy']:.4f}")
print(f"F1-Score Ponderado: {metrics['f1_weighted']:.4f}")
print(f"F1-Score Macro:     {metrics['f1_macro']:.4f}")
print(f"Recall (Desertor):  {metrics['recall_desertor']:.4f}")

fig_cm, axes_cm = plot_confusion_matrices(metrics['confusion_matrix'])
plt.show()

# Auditoria de Desempeno por Subgrupo (Sesgo y Equidad)
print('=== DESGLOSE DE RECALL (DESERTOR) POR SUBGRUPO ===')
subgroup_df = compute_subgroup_recall(X_test, y_test, y_pred)
display(subgroup_df.style.format({'Recall (Desertor)': '{:.4f}'}))
